# Chapter 4 (part 2) — Factor (Categorical) Variables in Regression

Covers the Ch 4 section on factor variables: Dummy Variable Representation → Factor Variables with Many Levels → Ordered Factor Variables, on Ames Housing.

The thread: **a regression only understands numbers, so every category gets turned into numbers by *some* coding — and the coding decides what the coefficients mean.** One-hot dummies turn "which group" into "how far from a baseline group". Many-level factors strain that machinery, because rare levels have almost no data behind their coefficients. Ordered factors tempt you to use integers, which quietly assumes equal gaps between levels.

A data note: this version of Ames stores quality ratings as *words* (`Overall_Qual` is `Very_Poor` … `Very_Excellent`, not 1-10), and uses explicit levels like `No_Garage` / `No_Basement` instead of missing values — so you'll make the integer mapping yourself and decide what a "no basement" house is worth on a quality scale.

Ask for a hint if you get stuck; the first couple of asks per question get a nudge, not the answer. The exam at the bottom is the intuition check.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape

## 1. Dummy Variable Representation

To feed a category into a regression you make one 0/1 column per level ("is this house a duplex? yes=1/no=0") and leave one level out as the baseline. Numeric picture: suppose average prices are $200k for single-family, $150k for duplexes, $140k for townhouses. Leave single-family as the baseline: the intercept is $200k (the average of the baseline group), the duplex coefficient is -$50k, the townhouse coefficient is -$60k. A duplex's predicted price = 200 + (-50) = $150k, exactly its group average. With only a category in the model, the regression just reproduces the group means.

**Question 1.** Fit `Sale_Price ~ C(Bldg_Type)`. (a) Which level did `statsmodels` use as the baseline? Verify with `groupby('Bldg_Type').Sale_Price.mean()` that the intercept equals that group's mean and that each coefficient equals the group's mean minus the baseline mean. (b) Pick the largest coefficient and its standard error. Why are some levels' standard errors much bigger than others? (c) The model's R² is what fraction? What does that R² mean in the language of "variation within vs. between groups"? (You saw the same quantity in the ANOVA notebook — what's the connection?)

*Tools for this question:*
- `C(col)` inside a formula treats `col` as categorical (one dummy per level except the first alphabetically). Coefficient names look like `C(Bldg_Type)[T.Duplex]`.
- `df.groupby("col")["y"].mean()` for group means.

_Your interpretation:_

## 2. Choosing the Reference Level, and the Dummy-Variable Trap

The baseline is arbitrary — it's a choice about *which comparison is most readable*, not a modeling decision. Changing it shuffles the coefficients but leaves every prediction (and R²) identical. Numeric picture: with baselines $200k (single-family), duplex -$50k, townhouse -$60k, switch the baseline to townhouse: intercept becomes $140k, single-family becomes +$60k, duplex becomes +$10k. Same three predictions: 200, 150, 140.

The **dummy-variable trap**: if you give the model an intercept *and* a 0/1 column for every level, the dummy columns add up to exactly the intercept column (every house is in exactly one group), so the model can't tell how much credit to give each — infinitely many coefficient sets fit equally well.

**Question 2.** (a) Re-fit the Question 1 model with `TwnhsE` as the baseline: `C(Bldg_Type, Treatment(reference='...'))`. Which numbers changed, which didn't (compare R², fitted values for a few houses, and the F-statistic)? (b) Build the full 5-column dummy matrix yourself with `pd.get_dummies` (keep all levels), add a constant, and compute its rank with `np.linalg.matrix_rank`. How does it compare to the number of columns, and what does that say about the trap? (c) Try fitting that matrix with `sm.OLS` anyway. What does `statsmodels` do or warn about?

*Tools for this question:*
- `C(col, Treatment(reference="TwnhsE"))` sets the baseline level.
- `pd.get_dummies(df["col"], dtype=int)` keeps every level; `sm.add_constant(M)`; `np.linalg.matrix_rank(M)`; `sm.OLS(y, X).fit()` (the non-formula API — it adds no intercept on its own).

_Your interpretation:_

## 3. Dummies Alongside a Numeric Predictor

Once a numeric predictor is in the model, a dummy coefficient no longer compares *group averages* — it compares groups **at the same value of the numeric predictor**. Numeric picture: duplexes average $150k and are on average 2,000 sqft; single-family homes average $200k at 1,500 sqft. Raw gap = -$50k. But the duplexes are bigger; if each sqft is worth $100, a 2,000 sqft house "should" be worth $50k more, so *at equal size* the duplex premium is not -$50k but -$100k. Same coefficient label, different question.

**Question 3.** Fit `Sale_Price ~ Gr_Liv_Area + C(Bldg_Type)`. (a) Compare each Bldg_Type coefficient to Question 1 — which ones changed the most, and in which direction? Use group-level mean `Gr_Liv_Area` by `Bldg_Type` to explain why. (b) The model forces every building type to have the *same* price-per-sqft slope and only shifts the line up or down. Plot the fitted lines against the data for two building types. Do you believe parallel lines are reasonable here? (Hold the thought — interactions come in the next notebook.)

*Tools for this question:*
- `res.predict(new_df)` gives fitted values for hand-built rows (e.g., two building types at the same `Gr_Liv_Area`); `df.groupby("Bldg_Type")["Gr_Liv_Area"].mean()`.

_Your interpretation:_

## 4. Factor Variables with Many Levels

`Neighborhood` has 28 levels, from North Ames (443 homes) down to Landmark (1 home). A dummy for a one-house neighborhood fits that house *perfectly*: its coefficient is exactly that house's price gap, with essentially nothing to tell signal from noise. The book's remedy is to consolidate levels using the data: fit a first model without the factor, look at the residuals grouped by level, and merge levels with similar residuals into a few bins.

Leakage warning: if you compute those residual-based bins on all data, then evaluate on the same data, the grouping has already "seen" the answers. The honest version computes the groupings on the *training* split only and applies them to the test split.

**Question 4.** (a) Fit `Sale_Price ~ C(Neighborhood)` and list the coefficients and standard errors for the five smallest neighborhoods. What do you see? (b) Split train/test (70/30, fixed seed). On the **train set only**: fit `Sale_Price ~ Gr_Liv_Area + Year_Built`, compute the median residual per neighborhood, and cut the neighborhoods into 5 tiers by those medians (`pd.qcut` on the neighborhood-level medians, or sort and chop). (c) Build three models on train and compare **test RMSE**: (i) no neighborhood information, (ii) 28 neighborhood dummies, (iii) 5-tier factor. Which wins on test and why? What would you check about neighborhoods that appear in test but not in train? (d) The tiers are a coarse stand-in for a continuous quantity. What is that quantity, and why is it a fair thing for the model to know?

*Tools for this question:*
- `df["Neighborhood"].value_counts()`; `df.assign(resid=res.resid).groupby("Neighborhood")["resid"].median()`.
- `pd.qcut(series, q=5, labels=False)` cuts into 5 equal-count bins; `df["col"].map(dict_or_series)` to attach tiers; unseen test levels map to `NaN` — handle with `.fillna(...)`.

_Your interpretation:_

## 5. Ordered Factor Variables

`Overall_Qual` runs from `Very_Poor` to `Very_Excellent` — ten ordered words. Two natural encodings: integers 1-10 (one slope: "each step up in quality is worth $X") or dummies (nine independent steps). The integer version assumes **equal gaps**; the dummy version assumes nothing but pays in parameters.

**Question 5.** (a) Make an integer column by mapping the ten level names to 1-10 (write the dictionary by hand — check the exact spellings with `value_counts()`). (b) Fit `Sale_Price ~ qual_int` and `Sale_Price ~ C(Overall_Qual)`. Plot mean `Sale_Price` by quality level against the straight line from the integer model. Where does the line fail? (c) Compute the *step sizes* — mean price at level k+1 minus mean at level k — for all 9 steps. Are they equal? Is the pattern what you'd expect from how people value quality? (d) Try `np.log(Sale_Price)` as the target. Do the step sizes look more uniform now? Why would a *percentage* effect per quality step be a more natural assumption than a dollar one? (e) Which model would you use if you wanted (i) a one-sentence interpretation for a stakeholder, (ii) the best predictions?

*Tools for this question:*
- `series.map({"Poor": 2, ...})` for the integer coding (check spellings with `value_counts()`); `df.groupby("col")["y"].mean().diff()` gives the step sizes.

_Your interpretation:_

## 6. When "None" Is a Real Category

Several columns here use a level like `No_Basement` or `No_Garage` where the original data would have missing values. `Bsmt_Qual` has levels `Poor, Fair, Typical, Good, Excellent` plus `No_Basement` (80 homes). If you map `Bsmt_Qual` to an integer scale, where do those 80 houses go?

**Question 6.** (a) Compare the mean `Sale_Price` of `No_Basement` houses to each `Bsmt_Qual` level (also check `Total_Bsmt_SF` for those homes). Where does "none" fall on the quality ordering by price — and would giving it the integer 0 be a defensible choice? (b) Propose a better encoding (hint: separate "has a basement" from "how good is it") and say how the interpretation of the quality coefficient changes. Don't just add code — explain what question each coefficient would answer. (c) Why might it be dangerous to *impute* a quality for these homes (e.g., the median)?

*Tools for this question:*
- `df.groupby("Bsmt_Qual")[["Sale_Price", "Total_Bsmt_SF"]].mean()`; `(df["Bsmt_Qual"] != "No_Basement").astype(int)` builds a 0/1 flag.

_Your interpretation:_

---
# Exam — check your intuition

7 questions. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, a few bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Reading dummy coefficients
*Role lean: DS*

Here is `Sale_Price ~ C(Foundation)` on all Ames homes. (`statsmodels` silently picked `BrkTil` as the reference level because it comes first alphabetically.)

```text
                             coef  std err     t   P>|t|
Intercept                128107.0   3847.0  33.0  <0.001
C(Foundation)[T.CBlock]   20177.0   4301.0   5.0  <0.001
C(Foundation)[T.PConc]    98962.0   4279.0  23.0  <0.001
C(Foundation)[T.Slab]    -17650.0  10427.0  -2.0   0.091
C(Foundation)[T.Stone]    21680.0  20812.0   1.0   0.298
C(Foundation)[T.Wood]     52793.0  30581.0   2.0   0.084

R-squared = 0.280
Homes per foundation: PConc 1310, CBlock 1244, BrkTil 311, Slab 49, Stone 11, Wood 5
```

(a) What is the average sale price of **Slab** homes, and how do you know from the table alone? (b) Slab's coefficient is -17,650 and Wood's is +52,793, yet *neither* has a small p-value while CBlock's +20,177 does. How can a difference of similar or larger size (Wood) be "less significant"? (c) Without re-fitting: if we change the reference level to `PConc`, what will the new intercept be, what will the `CBlock` coefficient become, and what will R-squared be? (d) Would changing the reference level change any *prediction* the model makes? Why do people still care which level is the reference?

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — The dummy-variable trap
*Role lean: DS · ML · interview*

Foundation has 6 levels. (a) Suppose you build 6 dummy columns (one per level) and **also** keep an intercept. Why does ordinary least squares break (or give nonsense) — use a concrete row, e.g. a PConc house, to say what the columns add up to. (b) Suppose instead you keep all 6 dummies and **drop the intercept**. What does each coefficient now equal, and what has to be different about how you read R-squared in that model? (c) Interview version: "Why does one-hot encoding for linear regression drop a column, but a random forest doesn't care?" Answer in two or three sentences.

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — A dummy coefficient is a conditional comparison
*Role lean: Causal · DS*

Two models for the garage type of a home. In Model B the `No_Garage` dummy gets a coefficient of about +26,256, i.e. houses with no garage are predicted to sell for *more* than comparable attached-garage homes.

```text
MODEL A: Sale_Price ~ C(Garage_Type)
                                           coef  std err      t   P>|t|
Intercept                              203772.0   1666.0  122.0  <0.001
C(Garage_Type)[T.Basment]              -53299.0  11671.0   -5.0  <0.001
C(Garage_Type)[T.BuiltIn]               45572.0   5348.0    9.0  <0.001
C(Garage_Type)[T.CarPort]              -98206.0  17973.0   -5.0  <0.001
C(Garage_Type)[T.Detchd]               -71304.0   2986.0  -24.0  <0.001
C(Garage_Type)[T.More_Than_Two_Types]  -49133.0  14548.0   -3.0  <0.001
C(Garage_Type)[T.No_Garage]            -98823.0   5777.0  -17.0  <0.001

MODEL B: Sale_Price ~ Garage_Area + C(Garage_Type)
                                           coef  std err     t   P>|t|
Intercept                               78693.0   3365.0  23.0  <0.001
C(Garage_Type)[T.Basment]              -30803.0   9360.0  -3.0   0.001
C(Garage_Type)[T.BuiltIn]               32888.0   4293.0   8.0  <0.001
C(Garage_Type)[T.CarPort]              -71260.0  14404.0  -5.0  <0.001
C(Garage_Type)[T.Detchd]               -45993.0   2471.0 -19.0  <0.001
C(Garage_Type)[T.More_Than_Two_Types] -110053.0  11743.0  -9.0  <0.001
C(Garage_Type)[T.No_Garage]             26256.0   5562.0   5.0  <0.001
Garage_Area                               238.0      6.0  40.0  <0.001

All 157 'No_Garage' homes have Garage_Area = 0. Reference level = Attchd.
Homes per type: Attchd 1731, Detchd 782, BuiltIn 186, No_Garage 157, Basment 36, More_Than_Two_Types 23, CarPort 15
```

(a) Compare the `Detchd` coefficient in Model A to Model B. Why did it shrink in size, and what *question* does each coefficient answer? (b) The `No_Garage` coefficient is positive in Model B but those homes sell for ~$99k *less* on average than `Attchd` homes (see the raw averages implied by Model A). Explain how both can be true, paying attention to where in `Garage_Area` the comparison is being made and whether there are any `Attchd` homes there. (c) Would you report the No_Garage coefficient from Model B to a stakeholder? What would you change in the model specification to make it meaningful?

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — Factors with many levels
*Role lean: DS · ML*

`Exterior_1st` has 16 levels. This is a slice of `Sale_Price ~ C(Exterior_1st)` showing only the seven rarest levels (reference level is `AsbShng`, 44 homes):

```text
         coef vs AsbShng  std err  homes
ImStucc         160499.0  73799.0    1.0
PreCast         158499.0  73799.0    1.0
CBlock           -8251.0  52761.0    2.0
AsphShn         -10314.0  52761.0    2.0
Stone           156999.0  52761.0    2.0
BrkComm          35999.0  31758.0    6.0
Stucco           48259.0  15648.0   43.0
```

(a) Why are the standard errors so large for the levels with 1-2 homes, and what, concretely, is the model doing to fit a 1-home level perfectly? (b) Name two reasonable ways to handle a factor like this before putting it in a regression (think: what do you do with the rare levels, and what do you do about levels that behave similarly). (c) A colleague proposes replacing the factor with "the mean `Sale_Price` of each exterior type" as one numeric column. Name one big advantage and one serious danger of that approach.

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — Treating an ordered factor as a number
*Role lean: DS · ML · interview*

`Kitchen_Qual` is ordered (Fair < Typical < Good < Excellent). Someone codes it 1-4 and fits a single slope; you can compare to the model with one dummy per level:

```text
Model: Sale_Price ~ kq   (the single numeric column kq; 1 'Poor' home dropped)
Intercept = -23,456   slope on kq = 81,317   R-squared = 0.454

Kitchen_Qual  coded as  homes  actual mean price  linear-coding fitted
        Fair         1     70             105907                 57861
     Typical         2   1494             139550                139177
        Good         3   1160             210836                220494
   Excellent         4    205             337339                301810

Same data with one dummy per level (C(Kitchen_Qual)): R-squared = 0.482
```

(a) What does the linear coding force to be true about the gaps between adjacent quality levels, and where in the table can you see that it's not true? (b) The dummy model has higher R-squared with 2 extra parameters. Is that an argument for the dummies, against, or neither — and what would you check to decide? (c) Interview version: "For ordinal features, when would you choose integer coding, one-hot, or something else, and does your answer change between linear regression and gradient-boosted trees?" 

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_

### Exam Q6 — Target encoding and leakage (supplementary)
*Role lean: ML · DS · interview*

A high-cardinality factor like `Neighborhood` (28 levels) is converted into one numeric column: each house gets the **mean Sale_Price of its neighborhood**. Two ways to compute those means, each used as the *prediction itself* for 500 held-out homes (averaged over 30 random splits; the 'train-only' means are computed from just 150 homes):

```text
neighborhood means computed from ALL 2,930 homes (including the held-out ones):   held-out R-squared = 0.585
neighborhood means computed only from the 150 training homes:                       held-out R-squared = 0.470
```
(a) Why is the first number higher, and which one tells you how the feature would do on genuinely new data? (b) Why would the gap be even *bigger* if the neighborhoods were smaller (say, 3 homes each)? (c) Describe in two or three sentences how you'd build this feature correctly inside cross-validation, and one extra trick people use to stop small groups from being over-trusted.

**Your answer (Q6):**

> _write here_

**Review (Q6):**

> _pending — filled in after you answer_

### Exam Q7 — Choosing an encoding for the job
*Role lean: DS · Causal · ML*

You're asked three questions about Ames. For each, say what encoding you'd use for `Neighborhood` (28 levels, some with fewer than 10 homes), and one thing you'd check: (1) a stakeholder asks "how much more do homes in Northridge Heights sell for than in Old Town, all else equal?"; (2) you're building a price model whose only job is accurate predictions on new sales; (3) you want to *control* for location while estimating the effect of central air on price, and you don't care about the neighborhood coefficients themselves. Explain how the best encoding changes with the goal.

**Your answer (Q7):**

> _write here_

**Review (Q7):**

> _pending — filled in after you answer_